In [1]:
from __future__ import annotations

import base64
import json
import time
import uuid
from dataclasses import dataclass
from typing import Any

from cryptography.exceptions import InvalidSignature
from cryptography.hazmat.primitives.asymmetric.ed25519 import (
    Ed25519PrivateKey,
    Ed25519PublicKey,
)
from cryptography.hazmat.primitives import serialization


FORMAT = "LIC1"
ALGORITHM = "Ed25519"


def b64e(data: bytes) -> str:
    return base64.urlsafe_b64encode(data).rstrip(b"=").decode("ascii")


def b64d(value: str) -> bytes:
    if not isinstance(value, str):
        raise ValueError("Expected string")

    # Reject characters outside Base64URL alphabet.
    if any(c not in "ABCDEFGHIJKLMNOPQRSTUVWXYZ"
           "abcdefghijklmnopqrstuvwxyz"
           "0123456789-_=" for c in value):
        raise ValueError("Invalid Base64URL")

    padding = "=" * (-len(value) % 4)
    return base64.urlsafe_b64decode(value + padding)


def canonical_json(obj: dict[str, Any]) -> bytes:
    """
    Deterministic JSON representation.

    Important: every signer/verifier must use exactly the same
    canonicalization rules.
    """
    return json.dumps(
        obj,
        sort_keys=True,
        separators=(",", ":"),
        ensure_ascii=False,
    ).encode("utf-8")


def generate_keypair() -> tuple[bytes, bytes]:
    """
    Returns:
        private_key_pem
        public_key_pem
    """
    private_key = Ed25519PrivateKey.generate()
    public_key = private_key.public_key()

    private_pem = private_key.private_bytes(
        encoding=serialization.Encoding.PEM,
        format=serialization.PrivateFormat.PKCS8,
        encryption_algorithm=serialization.NoEncryption(),
    )

    public_pem = public_key.public_bytes(
        encoding=serialization.Encoding.PEM,
        format=serialization.PublicFormat.SubjectPublicKeyInfo,
    )

    return private_pem, public_pem


def load_private_key(pem: bytes) -> Ed25519PrivateKey:
    key = serialization.load_pem_private_key(
        pem,
        password=None,
    )

    if not isinstance(key, Ed25519PrivateKey):
        raise TypeError("Not an Ed25519 private key")

    return key


def load_public_key(pem: bytes) -> Ed25519PublicKey:
    key = serialization.load_pem_public_key(pem)

    if not isinstance(key, Ed25519PublicKey):
        raise TypeError("Not an Ed25519 public key")

    return key


def issue_license(
    private_key_pem: bytes,
    *,
    user_id: str,
    name: str
) -> str:
    """
    Creates a stateless signed license.

    The returned license contains no secret.
    """

    if not user_id:
        raise ValueError("User id Required")

    if not name:
        raise ValueError("Name required")


    now = int(time.time())


    payload = {
        "format": FORMAT,
        "algorithm": ALGORITHM,
        "user_id": user_id,
        "license_id": str(uuid.uuid4()),
        "name": name,
        "issued_at": now,
    }

    message = canonical_json(payload)

    private_key = load_private_key(private_key_pem)
    signature = private_key.sign(message)

    return f"{FORMAT}.{b64e(message)}.{b64e(signature)}"


@dataclass(frozen=True)
class VerifiedLicense:
    license_id: str
    name: str
    user_id: str
    issued_at: int

def verify_license(
    license_string: str,
    public_keys: dict[str, bytes],
    *,
      now: int | None = None,
    max_future_seconds: int = 300,
) -> VerifiedLicense:
    """
    Stateless verification.

    public_keys maps:
        key_id -> public_key_pem
    """

    if not isinstance(license_string, str):
        raise ValueError("License must be a string")

    parts = license_string.split(".")

    if len(parts) != 3:
        raise ValueError("Malformed license")

    fmt, payload_b64, signature_b64 = parts

    if fmt != FORMAT:
        raise ValueError("Unsupported license format")

    payload_bytes = b64d(payload_b64)
    signature = b64d(signature_b64)

    if len(signature) != 64:
        raise ValueError("Invalid Ed25519 signature length")

    try:
        payload = json.loads(payload_bytes.decode("utf-8"))
    except (UnicodeDecodeError, json.JSONDecodeError) as exc:
        raise ValueError("Invalid payload") from exc

    if not isinstance(payload, dict):
        raise ValueError("Payload must be an object")

    # Security-critical fields are explicitly validated.
    required = {
        "format",
        "algorithm",
        "user_id",
        "license_id",
        "name",
        "issued_at",
    }

    if set(payload.keys()) != required:
        raise ValueError("Unexpected or missing license fields")

    if payload["format"] != FORMAT:
        raise ValueError("Invalid payload format")

    if payload["algorithm"] != ALGORITHM:
        raise ValueError("Unsupported algorithm")

    user_id = payload["user_id"]

    if not isinstance(user_id, str):
        raise ValueError("Invalid key_id")

    public_key_pem = public_keys.get(user_id)

    if public_key_pem is None:
        raise ValueError("Unknown signing key")

    public_key = load_public_key(public_key_pem)

    # IMPORTANT:
    # Verify the signature over the exact original canonical payload.
    try:
        public_key.verify(signature, payload_bytes)
    except InvalidSignature as exc:
        raise ValueError("Invalid license signature") from exc

    current_time = int(time.time()) if now is None else now

 
    return VerifiedLicense(
        name=payload["name"],
        user_id=user_id,
        license_id = payload["license_id"],
        issued_at = payload["issued_at"]
    )


def has_feature(
    license: VerifiedLicense,
    feature: str,
) -> bool:
    return feature in license.features